# Frequency-Dependent Peripheral Nerve Stimulation — Multi-Scale Study

Single-axon biophysics (NEURON) → synaptic relay with short-term plasticity (Brian2) → downstream network dynamics (NAc / insula / CA3, Brian2).

**Pipeline:**
```
Level 1 (NEURON)          Level 2 (Brian2)              Level 3 (Brian2)
peripheral axon      →    NTS relay synapse        →    NAc / insula / CA3
(MRG or Sundt model)      (Tsodyks-Markram STP)         (population dynamics)
```

Run cells top to bottom. Each cell below corresponds to one module of the original project (`models/`, `stim/`, `sweep/`, `coupling/`, `network/`) but everything lives in this single notebook's namespace for Colab convenience — no local package imports needed.


## 1. Install dependencies

This installs NEURON and Brian2 from PyPI. Takes ~1-2 min on Colab.


In [1]:
!pip install -q neuron brian2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 22.6 MB/s eta 0:00:00


## 2. (Optional) Real MRG / Sundt mechanism files

By default this notebook falls back to Hodgkin-Huxley kinetics so everything
runs out of the box. For biophysically accurate results, get the real
mechanism files:

- **MRG** (McIntyre, Richardson, Grill 2002) — ModelDB accession **3810**
- **Sundt C-fiber** (Sundt, Gamper, Jaffe 2015) — ModelDB accession **189712**

Download from https://modeldb.science, extract the `.mod` files, then in
Colab: use the file-upload icon (left sidebar) to upload them into a
`mechanisms/` folder, then run:

```python
!mkdir -p mechanisms
# upload .mod files into mechanisms/ via the Colab file browser, then:
!cd mechanisms && nrnivmodl
```

Restart the runtime after compiling so NEURON picks up the new mechanisms.
Until then, the HH fallback below is fully functional for testing the pipeline.


In [2]:
import os
# Uncomment once you've uploaded .mod files into mechanisms/ and compiled them:
# os.system("cd mechanisms && nrnivmodl")


## 3. Imports

In [3]:
from neuron import h
import numpy as np
h.load_file("stdrun.hoc")


1.0

## 4. Extracellular field (`stim/extracellular_field.py`)

Point-source extracellular potential coupling (McNeal 1976 / Rattay 1986)
and a biphasic / sinusoidal waveform generator (sinusoidal used for KHFAC
block stimuli).


In [4]:
def point_source_potential(x, y, z, elec_pos, current, rho=300.0):
    """Extracellular potential (mV) at (x,y,z) [um] from a point current source."""
    ex, ey, ez = elec_pos
    r = np.sqrt((x - ex) ** 2 + (y - ey) ** 2 + (z - ez) ** 2)  # um
    r_cm = np.maximum(r * 1e-4, 1e-6)  # um -> cm, avoid singularity
    return (rho * current) / (4 * np.pi * r_cm)  # mV


def biphasic_waveform(freq_hz, amp, duration_ms, dt, pulse_width_ms=0.1,
                       interphase_ms=0.0, waveform="rectangular"):
    """Generate a periodic biphasic or sinusoidal stimulus waveform."""
    t = np.arange(0, duration_ms, dt)
    i_t = np.zeros_like(t)
    period_ms = 1000.0 / freq_hz

    if waveform == "sinusoidal":
        i_t = amp * np.sin(2 * np.pi * freq_hz * t / 1000.0)
        return t, i_t

    n_cycles = int(np.floor(duration_ms / period_ms))
    for n in range(n_cycles):
        t0 = n * period_ms
        pos_start, pos_end = t0, t0 + pulse_width_ms
        neg_start = pos_end + interphase_ms
        neg_end = neg_start + pulse_width_ms
        i_t[(t >= pos_start) & (t < pos_end)] = amp
        i_t[(t >= neg_start) & (t < neg_end)] = -amp

    return t, i_t


def apply_extracellular_field(section_list, coords, elec_pos, current, rho=300.0):
    """Set e_extracellular for each segment based on a point-source field."""
    for sec, (x, y, z) in zip(section_list, coords):
        v_ext = point_source_potential(x, y, z, elec_pos, current, rho)
        for seg in sec:
            seg.e_extracellular = v_ext


## 5. Myelinated axon model (`models/mrg_axon.py`)

MRG (McIntyre-Richardson-Grill) geometry. Auto-detects the real MRG
mechanism if compiled; otherwise uses an HH fallback.


In [5]:
MRG_PARAMS = {
    # fiber_diam: (node_diam, node_length, internode_length, n_myelin_lamellae)
    5.7: (1.9, 1.0, 500, 80),
    8.7: (2.8, 1.0, 750, 110),
    12.8: (3.4, 1.0, 1150, 130),
    16.0: (4.7, 1.0, 1400, 150),
}


def _has_mrg_mechanism():
    return hasattr(h, "axnode70") or hasattr(h, "MRGaxon")


class MRGAxon:
    def __init__(self, diameter_um=8.7, n_nodes=21):
        self.diameter = diameter_um
        self.n_nodes = n_nodes
        self.nodes = []
        self.internodes = []
        self.use_mrg = _has_mrg_mechanism()
        self._build()

    def _nearest_param_key(self):
        keys = np.array(list(MRG_PARAMS.keys()))
        return keys[np.argmin(np.abs(keys - self.diameter))]

    def _build(self):
        key = self._nearest_param_key()
        node_diam, node_len, inter_len, _ = MRG_PARAMS[key]

        for i in range(self.n_nodes):
            node = h.Section(name=f"node_{i}")
            node.diam = node_diam
            node.L = node_len
            node.nseg = 1
            node.insert("extracellular")

            if self.use_mrg:
                node.insert("axnode70")
            else:
                node.insert("hh")
                node.insert("pas")
                node.g_pas = 0.0001
                node.e_pas = -80

            self.nodes.append(node)

            if i < self.n_nodes - 1:
                inter = h.Section(name=f"internode_{i}")
                inter.diam = self.diameter * 0.7
                inter.L = inter_len
                inter.nseg = 6
                inter.insert("extracellular")
                inter.insert("pas")
                inter.g_pas = 1e-6
                inter.e_pas = -80
                self.internodes.append(inter)

        for i in range(self.n_nodes - 1):
            self.internodes[i].connect(self.nodes[i](1), 0)
            self.nodes[i + 1].connect(self.internodes[i](1), 0)

        if not self.use_mrg:
            print("[MRGAxon] WARNING: real MRG mechanism not found - using HH fallback. "
                  "Compile mechanisms/*.mod for accurate results.")

    def all_sections(self):
        return self.nodes + self.internodes

    def section_coords(self, orientation="z"):
        coords = []
        pos = 0.0
        for sec in self.all_sections():
            center = pos + sec.L / 2.0
            coords.append((0.0, 0.0, center) if orientation == "z" else (center, 0.0, 0.0))
            pos += sec.L
        return coords

    def record_last_node_spikes(self, threshold=-20):
        nc = h.NetCon(self.nodes[-1](0.5)._ref_v, None, sec=self.nodes[-1])
        nc.threshold = threshold
        spike_times = h.Vector()
        nc.record(spike_times)
        return spike_times


## 6. Unmyelinated C-fiber model (`models/sundt_cfiber.py`)

Sundt/Gamper/Jaffe-style single unmyelinated section. Auto-detects real
Sundt mechanisms if compiled; otherwise HH fallback.


In [6]:
def _has_sundt_mechanism():
    return hasattr(h, "nahh") or hasattr(h, "borgkdr")


class CFiber:
    def __init__(self, diameter_um=0.8, length_um=20000, seg_length_um=100):
        self.diameter = diameter_um
        self.length = length_um
        self.n_segs = max(1, int(length_um / seg_length_um))
        self.use_sundt = _has_sundt_mechanism()
        self._build()

    def _build(self):
        self.sec = h.Section(name="cfiber")
        self.sec.diam = self.diameter
        self.sec.L = self.length
        self.sec.nseg = self.n_segs
        self.sec.insert("extracellular")

        if self.use_sundt:
            self.sec.insert("nahh")
            self.sec.insert("borgkdr")
            self.sec.insert("pas")
        else:
            self.sec.insert("hh")
            self.sec.insert("pas")
            self.sec.g_pas = 0.0001
            self.sec.e_pas = -60

        if not self.use_sundt:
            print("[CFiber] WARNING: real Sundt mechanism not found - using HH fallback. "
                  "Compile mechanisms/*.mod for accurate results.")

    def section_coords(self, orientation="z"):
        coords = []
        step = self.sec.L / self.n_segs
        for i in range(self.n_segs):
            center = (i + 0.5) * step
            coords.append((0.0, 0.0, center) if orientation == "z" else (center, 0.0, 0.0))
        return coords

    def all_sections(self):
        return [self.sec]

    def record_end_spikes(self, threshold=-20):
        nc = h.NetCon(self.sec(1.0)._ref_v, None, sec=self.sec)
        nc.threshold = threshold
        spike_times = h.Vector()
        nc.record(spike_times)
        return spike_times


## 7. Threshold finder (`analysis/threshold_finder.py`)

Bisection search for activation or block threshold.


In [7]:
def find_threshold(run_trial_fn, amp_low, amp_high, tol=1e-4, max_iter=30):
    lo, hi = amp_low, amp_high
    if not run_trial_fn(hi):
        return None
    if run_trial_fn(lo):
        return lo
    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        if run_trial_fn(mid):
            hi = mid
        else:
            lo = mid
        if (hi - lo) < tol:
            break
    return hi


def verify_bracket(run_trial_fn, amp_low, amp_high, expand_factor=2.0, max_expansions=10):
    lo, hi = amp_low, amp_high
    for _ in range(max_expansions):
        if run_trial_fn(hi):
            return lo, hi
        hi *= expand_factor
    return lo, hi


## 8. Frequency sweep protocol (`sweep/frequency_sweep.py`)

Sweeps stimulation frequency (log-spaced, 1 Hz to 50 kHz) and finds
activation threshold (low freq) or conduction block threshold (kHz, KHFAC).

**Note:** running the FULL sweep (all frequencies × both fiber types ×
multiple diameters) is compute-heavy. On Colab's free CPU runtime, start
with a reduced frequency list (see the example call at the bottom) before
scaling up to the full range.


In [8]:
ELECTRODE_POS = (500.0, 0.0, 5000.0)  # um


def _run_activation_trial(fiber, amp, freq_hz, duration_ms=50, dt=0.005):
    coords = fiber.section_coords()
    sections = fiber.all_sections()
    spikes = (fiber.record_last_node_spikes() if hasattr(fiber, "record_last_node_spikes")
              else fiber.record_end_spikes())
    t_vec, i_vec = biphasic_waveform(freq_hz, amp, duration_ms, dt,
                                      pulse_width_ms=0.1, waveform="rectangular")
    h.dt = dt
    h.finitialize(-65)
    for i_t in i_vec:
        apply_extracellular_field(sections, coords, ELECTRODE_POS, i_t)
        h.fadvance()
    return spikes.size() > 0


def _run_block_trial(fiber, amp, freq_hz, duration_ms=60, dt=0.002,
                      test_pulse_amp=2.0, test_pulse_time_ms=50):
    coords = fiber.section_coords()
    sections = fiber.all_sections()
    spikes = (fiber.record_last_node_spikes() if hasattr(fiber, "record_last_node_spikes")
              else fiber.record_end_spikes())
    t_vec, i_vec = biphasic_waveform(freq_hz, amp, duration_ms, dt, waveform="sinusoidal")
    h.dt = dt
    h.finitialize(-65)
    test_delivered = False
    for k, t in enumerate(t_vec):
        i_t = i_vec[k]
        if t >= test_pulse_time_ms and not test_delivered:
            i_t += test_pulse_amp
            test_delivered = True
        apply_extracellular_field(sections, coords, ELECTRODE_POS, i_t)
        h.fadvance()
    post_test_spikes = [s for s in spikes if s >= test_pulse_time_ms]
    return len(post_test_spikes) == 0


def sweep_fiber(fiber_builder, diameter, frequencies_hz, mode="activation",
                 amp_bounds=(0.01, 5.0)):
    trial_fn = _run_activation_trial if mode == "activation" else _run_block_trial
    results = {}
    for freq in frequencies_hz:
        fiber = fiber_builder(diameter)
        lo, hi = amp_bounds
        lo, hi = verify_bracket(lambda a: trial_fn(fiber, a, freq), lo, hi)
        thresh = find_threshold(lambda a: trial_fn(fiber, a, freq), lo, hi)
        results[freq] = thresh
        print(f"  freq={freq:>8.1f} Hz  threshold={thresh}")
    return results


def default_frequency_range():
    low = np.array([1, 2, 5, 10, 20, 50, 100, 200, 500])
    high = np.array([1000, 2000, 5000, 10000, 20000, 50000])
    return np.concatenate([low, high])


def mrg_builder(diameter):
    return MRGAxon(diameter_um=diameter)


def cfiber_builder(diameter):
    return CFiber(diameter_um=diameter)


### Try a small sweep

Start small (a handful of frequencies) to check runtime before scaling to
`default_frequency_range()`.


In [9]:
# Quick test: activation threshold, MRG axon, 8.7 um, a few frequencies
test_freqs = [10, 50, 200]
results_mrg_activation = sweep_fiber(mrg_builder, diameter=8.7,
                                      frequencies_hz=test_freqs, mode="activation")
results_mrg_activation


[MRGAxon] WARNING: real MRG mechanism not found - using HH fallback. Compile mechanisms/*.mod for accurate results.
  freq=    10.0 Hz  threshold=None
[MRGAxon] WARNING: real MRG mechanism not found - using HH fallback. Compile mechanisms/*.mod for accurate results.
  freq=    50.0 Hz  threshold=1.9532035827636716
[MRGAxon] WARNING: real MRG mechanism not found - using HH fallback. Compile mechanisms/*.mod for accurate results.
  freq=   200.0 Hz  threshold=1.9532035827636716


{10: None, 50: 1.9532035827636716, 200: 1.9532035827636716}

## 9. Brian2: spike bridge, NTS relay (TM plasticity), downstream regions

Level 1 output (NEURON spike times) feeds Level 2 (NTS relay with
Tsodyks-Markram short-term plasticity) feeds Level 3 (NAc / insula / CA3).


In [10]:
from brian2 import (NeuronGroup, Synapses, SpikeGeneratorGroup, SpikeMonitor,
                     run, ms, mV, nS, defaultclock)


def neuron_spikes_to_brian_group(spike_times_ms, n_fibers=1, fiber_index=0):
    times = np.array(list(spike_times_ms)) * ms
    indices = np.full(len(times), fiber_index)
    return SpikeGeneratorGroup(n_fibers, indices, times)


def merge_fiber_populations(spike_dict_ms):
    fiber_ids = list(spike_dict_ms.keys())
    n_fibers = len(fiber_ids)
    all_times, all_indices = [], []
    for idx, fid in enumerate(fiber_ids):
        for t in spike_dict_ms[fid]:
            all_times.append(t)
            all_indices.append(idx)
    order = np.argsort(all_times)
    times = np.array(all_times)[order] * ms
    indices = np.array(all_indices)[order]
    return SpikeGeneratorGroup(n_fibers, indices, times), fiber_ids


In [11]:
NTS_EQS = """
dv/dt = (v_rest - v + g_syn*(E_syn - v)/gL) / tau_m : volt (unless refractory)
dg_syn/dt = -g_syn / tau_syn : siemens
v_rest : volt
E_syn : volt
gL : siemens
tau_m : second
tau_syn : second
"""

TM_SYN_EQS = """
w : siemens
U : 1
tau_f : second
tau_d : second
du/dt = -u / tau_f : 1 (event-driven)
dx/dt = (1 - x) / tau_d : 1 (event-driven)
"""

TM_ON_PRE = """
u += U * (1 - u)
g_syn_post += w * u * x
x -= u * x
"""


def build_nts_relay(n_nts_neurons=50, synapse_type="depressing", base_w=25 * nS):
    """NOTE: base_w tuned so a single afferent fiber's sparse output can
    actually drive NTS neurons to threshold -- see Section 11 discussion
    of why the original 2 nS default produced total silence."""
    nts = NeuronGroup(n_nts_neurons, NTS_EQS, threshold="v > -50*mV",
                       reset="v = -65*mV", refractory=2 * ms, method="euler")
    nts.v = -65 * mV
    nts.v_rest = -65 * mV
    nts.E_syn = 0 * mV
    nts.gL = 10 * nS
    nts.tau_m = 20 * ms
    nts.tau_syn = 5 * ms

    if synapse_type == "depressing":
        U_val, tau_f_val, tau_d_val = 0.5, 20 * ms, 700 * ms
    else:
        U_val, tau_f_val, tau_d_val = 0.15, 600 * ms, 50 * ms

    def make_synapses(source_group):
        syn = Synapses(source_group, nts, model=TM_SYN_EQS, on_pre=TM_ON_PRE, method="euler")
        syn.connect(p=0.5)
        syn.w = base_w
        syn.U = U_val
        syn.tau_f = tau_f_val
        syn.tau_d = tau_d_val
        syn.x = 1.0
        syn.u = U_val
        return syn

    return nts, make_synapses


In [12]:
LIF_EQS = """
dv/dt = (v_rest - v + g_syn*(E_syn - v)/gL) / tau_m : volt (unless refractory)
dg_syn/dt = -g_syn / tau_syn : siemens
v_rest : volt
E_syn : volt
gL : siemens
tau_m : second
tau_syn : second
"""


def _build_lif_population(n, tau_m, v_rest=-65 * mV, refractory=3 * ms):
    pop = NeuronGroup(n, LIF_EQS, threshold="v > -50*mV", reset="v = -65*mV",
                       refractory=refractory, method="euler")
    pop.v = v_rest
    pop.v_rest = v_rest
    pop.E_syn = 0 * mV
    pop.gL = 10 * nS
    pop.tau_m = tau_m
    pop.tau_syn = 5 * ms
    return pop


def build_nac(n=100):
    """NAc: reached via NTS -> LC / VTA-adjacent dopaminergic modulation."""
    return _build_lif_population(n, tau_m=25 * ms)


def build_insula(n=100):
    """Insula: interoceptive relay (thalamic stage omitted for now)."""
    return _build_lif_population(n, tau_m=20 * ms)


def build_ca3(n=200, recurrent_p=0.1, recurrent_w=0.6 * nS):
    """CA3: recurrent collateral network.

    NOTE: recurrent_w was originally 1.5 nS with no refractory period on
    the population -- that combination produces runaway self-sustaining
    excitation (no inhibition in this reduced model to check it), which
    showed up as CA3 saturating near the simulation's maximum possible
    firing rate regardless of stimulation frequency. 0.6 nS + a 3 ms
    refractory keeps CA3 in a regime where recurrent amplification is
    visible without exploding. If you add an inhibitory population later,
    recurrent_w can likely go back up without instability.
    """
    pop = _build_lif_population(n, tau_m=20 * ms)
    recurrent_syn = Synapses(pop, pop, on_pre="g_syn_post += w", model="w : siemens")
    recurrent_syn.connect(condition="i != j", p=recurrent_p)
    recurrent_syn.w = recurrent_w
    return pop, recurrent_syn


def connect_region(source_group, target_pop, p=0.2, w=2 * nS):
    syn = Synapses(source_group, target_pop, on_pre="g_syn_post += w_syn",
                    model="w_syn : siemens")
    syn.connect(p=p)
    syn.w_syn = w
    return syn


## 10. End-to-end example: axon → NTS relay → CA3

Verifies the pipeline connects correctly at a single test frequency.


In [13]:
def run_level1(freq_hz=20, amp=1.0, duration_ms=200, dt=0.005):
    fiber = MRGAxon(diameter_um=8.7, n_nodes=15)
    coords = fiber.section_coords()
    sections = fiber.all_sections()
    spikes = fiber.record_last_node_spikes()
    elec_pos = (500.0, 0.0, 3000.0)
    t_vec, i_vec = biphasic_waveform(freq_hz, amp, duration_ms, dt, pulse_width_ms=0.1)
    h.dt = dt
    h.finitialize(-65)
    for i_t in i_vec:
        apply_extracellular_field(sections, coords, elec_pos, i_t)
        h.fadvance()
    return list(spikes)


def run_levels_2_3(spike_times_ms, duration_ms=200):
    afferent = neuron_spikes_to_brian_group(spike_times_ms, n_fibers=1, fiber_index=0)
    nts, make_synapses = build_nts_relay(n_nts_neurons=30, synapse_type="depressing")
    nts_syn = make_synapses(afferent)
    ca3_pop, ca3_recurrent = build_ca3(n=100)
    relay_to_ca3 = connect_region(nts, ca3_pop, p=0.25)
    nts_mon = SpikeMonitor(nts)
    ca3_mon = SpikeMonitor(ca3_pop)
    defaultclock.dt = 0.1 * ms
    run(duration_ms * ms)
    return nts_mon, ca3_mon


print("Level 1: running MRG axon at 20 Hz...")
spikes = run_level1(freq_hz=20, amp=1.0)
print(f"  {len(spikes)} spikes generated at the distal node")

if len(spikes) == 0:
    print("  No spikes -- try increasing amp in run_level1() and re-run.")
else:
    print("Levels 2-3: NTS relay -> CA3 network...")
    nts_mon, ca3_mon = run_levels_2_3(spikes)
    print(f"  NTS spikes: {nts_mon.num_spikes}")
    print(f"  CA3 spikes: {ca3_mon.num_spikes}")


Level 1: running MRG axon at 20 Hz...
[MRGAxon] WARNING: real MRG mechanism not found - using HH fallback. Compile mechanisms/*.mod for accurate results.
  1 spikes generated at the distal node
Levels 2-3: NTS relay -> CA3 network...
  NTS spikes: 17
  CA3 spikes: 2


## 11. Multi-frequency, multi-region comparison

Does a given stimulation frequency show up differently in NAc vs. insula
vs. CA3? Each pathway gets its own synaptic dynamics so they're not
guaranteed to move in lockstep.

**If you ran an earlier version of this section and got curves that all
rose together with the same shape ("resonance" around 100 Hz for
everything) — that's a known failure mode of the naive version, not a
real result. Two things caused it, both fixed below:**

1. **A single afferent fiber drives the whole postsynaptic population
   identically and deterministically.** With no population averaging,
   every downstream neuron behaves as one unit — "on" or "off" together
   — which produces brittle, step-like curves that mostly just track how
   many input pulses arrived in the observation window, regardless of
   each pathway's actual plasticity. We now replicate a small **bundle**
   of fibers (representing a real nerve trunk with many similarly-tuned
   afferents) with independent timing jitter, so the population response
   is graded rather than all-or-none.
2. **Synaptic weights were strong enough that a single input spike alone
   crossed postsynaptic threshold.** That guarantees "more input pulses =
   proportionally more output" for every pathway, masking whatever
   frequency-selective gating the plasticity was supposed to produce. A
   facilitating synapse's entire purpose is to stay *below* threshold
   until repeated fast input builds it up — if one spike already does the
   job, that gate never gets exercised. Weights below are tuned so each
   pathway's threshold-crossing genuinely depends on its own plasticity
   dynamics, not just raw pulse count.

**Pathway design (the actual source of dissociation):**

- **NTS → NAc**: strongly facilitating (`U=0.05, tau_f=500 ms`) — needs
  several fast pulses within its facilitation window to build enough
  gain to cross threshold. Below that pace it stays essentially silent.
- **NTS → insula**: near-threshold single-pulse efficacy with mild
  depression (`U=0.4, tau_f=10 ms, tau_d=150 ms`) — responds from low
  frequency onward, more of a direct relay.
- **NTS → CA3**: moderate facilitation plus recurrent collaterals — behaves
  like insula at low frequency but pulls ahead at higher frequency as
  recurrent amplification kicks in.


In [14]:
from brian2 import start_scope, Network
import pandas as pd
import matplotlib.pyplot as plt

LIF_EQS_R = """
dv/dt = (v_rest - v + g_syn*(E_syn - v)/gL) / tau_m : volt (unless refractory)
dg_syn/dt = -g_syn / tau_syn : siemens
v_rest : volt
E_syn : volt
gL : siemens
tau_m : second
tau_syn : second
"""


def build_pop(n, tau_m, refractory=3 * ms):
    pop = NeuronGroup(n, LIF_EQS_R, threshold="v > -50*mV", reset="v = -65*mV",
                       refractory=refractory, method="euler")
    pop.v = -65 * mV
    pop.v_rest = -65 * mV
    pop.E_syn = 0 * mV
    pop.gL = 10 * nS
    pop.tau_m = tau_m
    pop.tau_syn = 5 * ms
    return pop


def build_ca3_r(n=100, recurrent_p=0.12, recurrent_w=1.0 * nS):
    """Recurrent collateral network. Keep recurrent_w modest -- push it much
    higher without adding inhibition and the network saturates to
    near-maximum firing (a runaway excitatory loop, not a real effect)."""
    pop = build_pop(n, tau_m=20 * ms)
    rec = Synapses(pop, pop, on_pre="g_syn_post += w", model="w : siemens")
    rec.connect(condition="i != j", p=recurrent_p)
    rec.w = recurrent_w
    return pop, rec


def bundle_from_single_fiber(spike_times_ms, n_fibers=20, jitter_ms=0.4, seed=0):
    """
    Replicate a single axon's spike train into a small bundle with per-fiber
    timing jitter -- approximates a real nerve trunk with many similarly-tuned
    afferents rather than one perfectly deterministic input. jitter_ms is kept
    small relative to the shortest inter-spike interval you'll test so fibers
    don't reorder into the same simulation timestep at high frequency.
    """
    rng = np.random.default_rng(seed)
    all_times, all_idx = [], []
    for fib in range(n_fibers):
        jittered = np.array(spike_times_ms) + rng.normal(0, jitter_ms, size=len(spike_times_ms))
        jittered = np.clip(jittered, 0, None)
        all_times.extend(jittered.tolist())
        all_idx.extend([fib] * len(spike_times_ms))
    order = np.argsort(all_times)
    times = np.array(all_times)[order] * ms
    idx = np.array(all_idx)[order]
    return SpikeGeneratorGroup(n_fibers, idx, times)


def make_tm_conn(source, target, U, tau_f, tau_d, p, w):
    syn = Synapses(source, target, model=TM_SYN_EQS, on_pre=TM_ON_PRE, method="euler")
    syn.connect(p=p)
    syn.w = w
    syn.U = U
    syn.tau_f = tau_f
    syn.tau_d = tau_d
    syn.x = 1.0
    syn.u = U
    return syn


In [15]:
def run_full_network(spike_times_ms, duration_ms, n_fibers=20):
    """
    Level 1 spikes -> fiber bundle -> NTS relay -> NAc / insula / CA3, each
    with its own pathway-specific plasticity. Explicit Network object (see
    Section 10 note) so nothing is silently dropped.
    """
    start_scope()

    afferent = bundle_from_single_fiber(spike_times_ms, n_fibers=n_fibers)

    nts = build_pop(30, tau_m=20 * ms, refractory=2 * ms)
    nts_syn = make_tm_conn(afferent, nts, U=0.5, tau_f=20 * ms, tau_d=700 * ms, p=0.3, w=6 * nS)

    nac_pop = build_pop(100, tau_m=30 * ms, refractory=3 * ms)
    insula_pop = build_pop(100, tau_m=20 * ms, refractory=3 * ms)
    ca3_pop, ca3_rec = build_ca3_r(100)

    c_nac = make_tm_conn(nts, nac_pop, U=0.05, tau_f=500 * ms, tau_d=100 * ms, p=0.3, w=9 * nS)
    c_insula = make_tm_conn(nts, insula_pop, U=0.4, tau_f=10 * ms, tau_d=150 * ms, p=0.3, w=7 * nS)
    c_ca3 = make_tm_conn(nts, ca3_pop, U=0.2, tau_f=150 * ms, tau_d=300 * ms, p=0.3, w=6 * nS)

    mons = {
        "NTS": SpikeMonitor(nts),
        "NAc": SpikeMonitor(nac_pop),
        "Insula": SpikeMonitor(insula_pop),
        "CA3": SpikeMonitor(ca3_pop),
    }

    defaultclock.dt = 0.1 * ms
    net = Network(afferent, nts, nts_syn, nac_pop, insula_pop, ca3_pop, ca3_rec,
                   c_nac, c_insula, c_ca3, *mons.values())
    net.run(duration_ms * ms)
    return mons


In [16]:
def cycles_to_duration_ms(freq_hz, n_cycles=8, min_ms=200, max_ms=1000):
    """Scale stimulus duration so low frequencies still get several cycles."""
    return float(min(max_ms, max(min_ms, n_cycles * 1000.0 / freq_hz)))


def run_one_frequency(freq_hz, amp=3.0, n_fibers=20):
    """Level 1 -> Levels 2/3 at a single frequency. Returns firing rates (Hz/neuron) per region."""
    duration_ms = cycles_to_duration_ms(freq_hz)
    spikes = run_level1(freq_hz=freq_hz, amp=amp, duration_ms=duration_ms)

    if len(spikes) == 0:
        return {"freq_hz": freq_hz, "duration_ms": duration_ms, "n_axon_spikes": 0,
                "NTS": 0.0, "NAc": 0.0, "Insula": 0.0, "CA3": 0.0}

    mons = run_full_network(spikes, duration_ms=duration_ms, n_fibers=n_fibers)

    n_neurons = {"NTS": 30, "NAc": 100, "Insula": 100, "CA3": 100}
    rates = {
        region: mon.num_spikes / n_neurons[region] / (duration_ms / 1000.0)
        for region, mon in mons.items()
    }
    rates["freq_hz"] = freq_hz
    rates["duration_ms"] = duration_ms
    rates["n_axon_spikes"] = len(spikes)
    return rates


### Run the sweep

~1-2 min on Colab CPU. Since each frequency runs a stochastic network only
once, results are noisy — average over several trials before treating any
single crossing point as exact (see the caveat box at the end).


In [17]:
test_frequencies = [1, 5, 10, 20, 50, 100, 200, 500]

records = []
for f in test_frequencies:
    print(f"Running {f} Hz...")
    rec = run_one_frequency(f, amp=3.0)
    records.append(rec)
    print(f"  axon spikes={rec['n_axon_spikes']}  "
          f"NTS={rec['NTS']:.2f} Hz  NAc={rec['NAc']:.2f} Hz  "
          f"Insula={rec['Insula']:.2f} Hz  CA3={rec['CA3']:.2f} Hz")

df = pd.DataFrame(records)
df


Running 1 Hz...
[MRGAxon] WARNING: real MRG mechanism not found - using HH fallback. Compile mechanisms/*.mod for accurate results.
  axon spikes=1  NTS=0.97 Hz  NAc=0.00 Hz  Insula=1.26 Hz  CA3=1.10 Hz
Running 5 Hz...
[MRGAxon] WARNING: real MRG mechanism not found - using HH fallback. Compile mechanisms/*.mod for accurate results.


KeyboardInterrupt: 

### Figure 1 — Region response curves across frequency

Firing rate per region vs. stimulation frequency (log-x axis). With the
bundle input and retuned pathways, NAc should stay near zero across most
of the range and only start rising at the high end, while insula and CA3
respond from low frequency — a genuine dissociation rather than three
copies of the same curve.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = {"NTS": "#888888", "NAc": "#d95f02", "Insula": "#1b9e77", "CA3": "#7570b3"}
for region in ["NTS", "NAc", "Insula", "CA3"]:
    ax.plot(df["freq_hz"], df[region], marker="o", label=region, color=colors[region], linewidth=2)
ax.set_xscale("log")
ax.set_xlabel("Stimulation frequency (Hz)")
ax.set_ylabel("Mean firing rate (Hz/neuron)")
ax.set_title("Region-specific frequency response")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()


### Figure 2 — Heatmap (region × frequency)

Same data, laid out so you can scan for a specific region/frequency
combination at a glance — e.g. "does 100 Hz reach CA3 but not NAc?"


In [ ]:
regions = ["NAc", "Insula", "CA3", "NTS"]
heat_data = df[regions].to_numpy().T  # rows=regions, cols=frequencies

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(heat_data, aspect="auto", cmap="viridis")
ax.set_yticks(range(len(regions)))
ax.set_yticklabels(regions)
ax.set_xticks(range(len(df["freq_hz"])))
ax.set_xticklabels(df["freq_hz"])
ax.set_xlabel("Stimulation frequency (Hz)")
ax.set_title("Region activity heatmap")
for i in range(len(regions)):
    for j in range(len(df["freq_hz"])):
        ax.text(j, i, f"{heat_data[i, j]:.1f}", ha="center", va="center",
                color="white" if heat_data[i, j] < heat_data.max() * 0.6 else "black", fontsize=8)
fig.colorbar(im, ax=ax, label="Hz/neuron")
plt.tight_layout()
plt.show()


### Figure 3 — Grouped bar chart per frequency

Easiest view for a direct region-vs-region comparison at each tested
frequency.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df))
width = 0.2
for i, region in enumerate(["NAc", "Insula", "CA3", "NTS"]):
    ax.bar(x + i * width, df[region], width, label=region, color=colors[region])
ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(df["freq_hz"])
ax.set_xlabel("Stimulation frequency (Hz)")
ax.set_ylabel("Mean firing rate (Hz/neuron)")
ax.set_title("Region comparison per frequency")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


### Dissociation table

Marks each region "active" if its rate exceeds a threshold (default 0.5 Hz/neuron
— adjust once weights are tuned against real data). This gives a direct,
readable answer to "does frequency X reach region Y?"


In [ ]:
ACTIVE_THRESHOLD_HZ = 0.5

dissociation = df.copy()
for region in ["NTS", "NAc", "Insula", "CA3"]:
    dissociation[region + "_active"] = dissociation[region] > ACTIVE_THRESHOLD_HZ

dissociation[["freq_hz", "NAc_active", "Insula_active", "CA3_active"]]


**Caveats — read before drawing conclusions:**

1. Weights and TM parameters are still hand-tuned to *produce* a clean
   dissociation for demonstration purposes, not fit to real
   electrophysiology. The specific frequency at which NAc turns on, or
   where CA3 pulls ahead of insula, is illustrative of the *mechanism*
   (facilitating vs. depressing vs. recurrent pathways genuinely gate
   differently), not a claim about the real vagus-NTS-NAc/insula/CA3
   circuit's actual operating point.
2. Each frequency is still a **single stochastic run**. Wrap
   `run_one_frequency` in a loop over a few trials and average before
   trusting any specific crossing point — bundle jitter and TM/connectivity
   randomness both introduce run-to-run variability.
3. `n_fibers=20` is a small stand-in bundle. A real vagal trunk carries far
   more afferents; if you want the population response to look smoother
   (less step-like), increasing `n_fibers` is the first thing to try,
   at the cost of runtime.
4. CA3's recurrent weight is deliberately kept low enough to avoid the
   runaway-saturation failure mode described in Section 10 — a more
   complete model would add feedback inhibition instead of just capping
   the excitatory gain by hand.


## Next steps

1. Compile the real MRG / Sundt mechanism files (Section 2) for biophysically accurate axon thresholds.
2. Scale the frequency sweep (Section 8) from the quick test to `default_frequency_range()`.
3. Add C-fiber runs alongside MRG (`cfiber_builder`) and build the selectivity index (ratio of C-fiber to A-fiber threshold per frequency).
4. Tune per-pathway synaptic weights and TM parameters (Section 11) against literature firing rates for NTS, NAc, insula, and CA3 — this is the single biggest lever on whether the dissociation pattern is quantitatively trustworthy.
5. Once weights are tuned, repeat the Section 11 sweep across a finer frequency grid and multiple trials per frequency (network stochasticity means a single run per frequency is noisy — average over ~5-10 trials before trusting a crossing point).
6. Consider adding the thalamic relay stage upstream of insula (Craig 2002 architecture) as an explicit intermediate population rather than a lumped direct connection.
